In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

In [2]:
conn = sqlite3.connect("../data/checking-logs.sqlite")

In [3]:
df = pd.read_sql_query(
    '''
    SELECT 
        uid,
        timestamp
    FROM checker
    WHERE 
        uid LIKE 'user_%'
        AND status = 'ready'
        AND labname = 'project1'
    ''',
    conn,
    parse_dates=['timestamp'],
)
df['date'] = df['timestamp'].dt.date
df

,uid,timestamp,date
0,user_4,2020-04-17 05:19:02.744528,2020-04-17
1,user_4,2020-04-17 05:22:45.549397,2020-04-17
2,user_4,2020-04-17 05:34:24.422370,2020-04-17
3,user_4,2020-04-17 05:43:27.773992,2020-04-17
4,user_4,2020-04-17 05:46:32.275104,2020-04-17
...,...,...,...
946,user_19,2020-05-15 10:22:39.698523,2020-05-15
947,user_19,2020-05-15 10:22:46.248162,2020-05-15
948,user_19,2020-05-15 10:23:18.043212,2020-05-15
949,user_28,2020-05-15 10:38:14.430013,2020-05-15


In [4]:
pivot_table = df.pivot_table(
  index='uid',
  columns='date',
  values='date',
  aggfunc='count',
  fill_value=0
)
pivot_table = pivot_table.cumsum(axis=1)
dates = list(pivot_table.columns)
users = pivot_table.index.tolist()

In [11]:
frames = []
for i in range(len(dates)):
    data = []
    for user in users:
        y_data = pivot_table.loc[user].iloc[:i+1].values
        x_data = list(range(1, i+2))
        data.append(go.Scatter(
            x=x_data,
            y=y_data,
            mode="lines+markers",
            name=user
        ))
    frames.append(go.Frame(data=data, name=str(i)))

initial_data = []
for user in users:
    initial_data.append(go.Scatter(
        x=[1],
        y=[pivot_table.loc[user].iloc[0]],
        mode="lines+markers",
        name=user
    ))

fig = go.Figure(
    data=initial_data,
    frames=frames
)

fig.update_layout(
    title=dict(
        text="Dynamic of commits per user in project1",
        font=dict(size=24),
        x=0.5
    ),
    xaxis=dict(
        range=[1, len(dates)+1],
        dtick=2
    ),

    width=1000,
    height=600,
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 300, "redraw": True}, "fromcurrent": True}]
                }
            ]
        }
    ]
)

fig.show()

In [6]:
conn.close()